In [ ]:
# ==========================================
# 🚀 DÍA 26: Q-LEARNING Y DQN (REINFORCEMENT LEARNING)
# ==========================================

# 1. Instalación de dependencias
!pip install -q gymnasium numpy matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
from collections import defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🎮 DÍA 26: REINFORCEMENT LEARNING - Q-LEARNING Y DQN")
print("="*70)

# ==========================================
# 2. TEORÍA RÁPIDA
# ==========================================
print("""
📚 CONCEPTOS CLAVE:

1. Agente: La IA que aprende (el "jugador")
2. Entorno: El mundo donde actúa (el "juego")
3. Estado (State): La situación actual
4. Acción (Action): Lo que el agente puede hacer
5. Recompensa (Reward): Premio/castigo por cada acción
6. Política (Policy): Estrategia del agente (qué hacer en cada estado)

🎯 Objetivo: Maximizar la recompensa total acumulada
""")

# ==========================================
# 3. Q-LEARNING CON FROZENLAKE
# ==========================================
print("\n" + "="*70)
print("🧊 PARTE 1: Q-LEARNING CON FROZENLAKE")
print("="*70)

print("""
🎮 FrozenLake:
- El agente debe llegar de S (Start) a G (Goal)
- Debe evitar los agujeros (H = Hole)
- El hielo es resbaladizo (acciones aleatorias)
- Recompensa: +1 al llegar a la meta, 0 en otros casos
""")

# Crear entorno
env_frozen = gym.make('FrozenLake-v1', is_slippery=True, render_mode='ansi')

print(f"\n📊 Entorno FrozenLake:")
print(f"   Espacio de estados: {env_frozen.observation_space.n} (4x4 = 16)")
print(f"   Espacio de acciones: {env_frozen.action_space.n} (0=Izq, 1=Abajo, 2=Der, 3=Arriba)")

# Visualizar el entorno (Llamando primero a .reset() para evitar el error ResetNeeded)
env_frozen.reset()
print("\n🗺️  Mapa del FrozenLake:")
print(env_frozen.render())

# ==========================================
# 4. IMPLEMENTAR Q-LEARNING
# ==========================================
print("\n" + "="*70)
print("🧠 IMPLEMENTANDO Q-LEARNING")
print("="*70)

print("""
📐 Algoritmo Q-Learning:
1. Inicializar Q-table (tabla de valores) en ceros
2. Por cada episodio:
   a. Observar estado actual
   b. Elegir acción (ε-greedy: explorar vs explotar)
   c. Ejecutar acción, observar recompensa y nuevo estado
   d. Actualizar Q-table:
      Q(s,a) = Q(s,a) + α * [r + γ * max(Q(s',a')) - Q(s,a)]
   e. Repetir hasta llegar a meta o caer en agujero
""")

# Hiperparámetros
EPISODES = 5000
ALPHA = 0.1        # Tasa de aprendizaje (learning rate)
GAMMA = 0.99       # Factor de descuento (importancia de recompensas futuras)
EPSILON = 1.0      # Exploración inicial (100%)
EPSILON_MIN = 0.01 # Exploración mínima
EPSILON_DECAY = 0.995  # Decaimiento de exploración

# Inicializar Q-table
q_table = defaultdict(lambda: np.zeros(env_frozen.action_space.n))

# Función para elegir acción (ε-greedy)
def choose_action(state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env_frozen.action_space.sample()  # Explorar
    else:
        return np.argmax(q_table[state])  # Explotar

# Entrenar agente
print(f"\n🚀 Entrenando agente por {EPISODES} episodios...")
rewards_history = []
success_rate = []

epsilon = EPSILON
for episode in range(EPISODES):
    state, _ = env_frozen.reset()
    total_reward = 0
    done = False

    while not done:
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env_frozen.step(action)
        done = terminated or truncated

        # Actualizar Q-table
        best_next_action = np.argmax(q_table[next_state])
        td_target = reward + GAMMA * q_table[next_state][best_next_action]
        td_error = td_target - q_table[state][action]
        q_table[state][action] += ALPHA * td_error

        state = next_state
        total_reward += reward

    rewards_history.append(total_reward)

    # Calcular tasa de éxito (ventana móvil de 100 episodios)
    if len(rewards_history) >= 100:
        recent_success = sum(rewards_history[-100:]) / 100
        success_rate.append(recent_success)

    # Decaer epsilon
    if epsilon > EPSILON_MIN:
        epsilon *= EPSILON_DECAY

    # Progreso
    if (episode + 1) % 1000 == 0:
        avg_reward = np.mean(rewards_history[-1000:])
        print(f"   Episodio {episode+1}/{EPISODES} | Recompensa promedio: {avg_reward:.3f} | Epsilon: {epsilon:.3f}")

print(f"\n✅ Entrenamiento completado")

# ==========================================
# 5. VISUALIZAR Q-TABLE APRENDIDA
# ==========================================
print("\n" + "="*70)
print("📊 Q-TABLE APRENDIDA")
print("="*70)

print("\n🗺️  Valores Q para cada estado (mejor acción en negrita):")
print("   0=Izq, 1=Abajo, 2=Der, 3=Arriba\n")

q_values_grid = np.zeros((4, 4, 4))
for state in range(16):
    row, col = state // 4, state % 4
    q_values_grid[row, col] = q_table[state]

for row in range(4):
    for col in range(4):
        state = row * 4 + col
        q_vals = q_table[state]
        best_action = np.argmax(q_vals)
        print(f"Estado {state:2d} ({row},{col}): ", end="")
        for action in range(4):
            if action == best_action:
                print(f"[{q_vals[action]:.3f}]", end=" ")
            else:
                print(f" {q_vals[action]:.3f} ", end=" ")
        print()

# ==========================================
# 6. GRÁFICAS DE APRENDIZAJE
# ==========================================
print("\n" + "="*70)
print("📈 VISUALIZACIÓN DEL APRENDIZAJE")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Recompensa por episodio
axes[0, 0].plot(rewards_history, alpha=0.3, color='blue', label='Recompensa por episodio')
axes[0, 0].plot(np.convolve(rewards_history, np.ones(100)/100, mode='valid'),
                color='red', linewidth=2, label='Promedio móvil (100)')
axes[0, 0].set_title('🎯 Recompensa por Episodio', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Episodio')
axes[0, 0].set_ylabel('Recompensa')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Tasa de éxito
if success_rate:
    axes[0, 1].plot(range(100, len(success_rate) + 100), success_rate, color='green', linewidth=2)
    axes[0, 1].axhline(y=0.7, color='red', linestyle='--', label='Meta: 70% éxito')
    axes[0, 1].set_title('📊 Tasa de Éxito (ventana de 100 episodios)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Episodio')
    axes[0, 1].set_ylabel('Tasa de éxito')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# 3. Q-values heatmap
im = axes[1, 0].imshow(q_values_grid[:, :, 1], cmap='YlOrRd', aspect='auto')
axes[1, 0].set_title('🔥 Q-values para acción "Abajo"', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Columna')
axes[1, 0].set_ylabel('Fila')
plt.colorbar(im, ax=axes[1, 0])

# 4. Política óptima
policy_grid = np.zeros((4, 4), dtype=int)
for state in range(16):
    row, col = state // 4, state % 4
    policy_grid[row, col] = np.argmax(q_table[state])

actions_symbols = ['←', '↓', '→', '↑']
for row in range(4):
    for col in range(4):
        axes[1, 1].text(col, row, actions_symbols[policy_grid[row, col]],
                       ha='center', va='center', fontsize=20, fontweight='bold',
                       color='blue')

axes[1, 1].set_xlim(-0.5, 3.5)
axes[1, 1].set_ylim(3.5, -0.5)
axes[1, 1].set_title('🧭 Política Óptima Aprendida', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(range(4))
axes[1, 1].set_yticks(range(4))
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 7. PROBAR AGENTE ENTRENADO
# ==========================================
print("\n" + "="*70)
print("🎮 PROBANDO AGENTE ENTRENADO")
print("="*70)

print("\n🚀 Ejecutando 10 episodios con política óptima (sin exploración):\n")

successes = 0
for episode in range(10):
    state, _ = env_frozen.reset()
    done = False
    path = [state]

    while not done:
        action = np.argmax(q_table[state])  # Siempre mejor acción
        next_state, reward, terminated, truncated, _ = env_frozen.step(action)
        done = terminated or truncated
        state = next_state
        path.append(state)

    if reward == 1:
        successes += 1
        print(f"✅ Episodio {episode+1}: ¡META ALCANZADA! Ruta: {path}")
    else:
        print(f"❌ Episodio {episode+1}: Cayó en agujero. Ruta: {path}")

print(f"\n📊 Tasa de éxito: {successes}/10 ({successes*10}%)")

# ==========================================
# 8. DQN CON CARTPOLE (VERSIÓN OPTIMIZADA)
# ==========================================
print("\n" + "="*70)
print("🎢 PARTE 2: DQN CON CARTPOLE")
print("="*70)

print("""
🎮 CartPole:
- Un carrito con un palo equilibrado
- El agente debe mover el carrito para mantener el palo vertical
- Recompensa: +1 por cada paso que el palo se mantenga en pie

💡 DQN Optimizado:
- Batch training (más eficiente)
- Menos episodios pero mejor rendimiento
""")

# Crear entorno CartPole
env_cart = gym.make('CartPole-v1')

print(f"\n📊 Entorno CartPole:")
print(f"   Espacio de estados: {env_cart.observation_space.shape} (continuo)")
print(f"   Espacio de acciones: {env_cart.action_space.n} (0=Izq, 1=Der)")

# ==========================================
# 9. IMPLEMENTAR DQN OPTIMIZADO
# ==========================================
print("\n" + "="*70)
print("🧠 IMPLEMENTANDO DQN OPTIMIZADO")
print("="*70)

from tensorflow.keras.layers import Input #type:ignore
from collections import deque

# Hiperparámetros (reducidos para velocidad)
EPISODES_DQN = 50  # ✅ Más rapido
GAMMA_DQN = 0.99
EPSILON_DQN = 1.0
EPSILON_MIN_DQN = 0.01
EPSILON_DECAY_DQN = 0.99
LEARNING_RATE = 0.001
MEMORY_SIZE = 2000
BATCH_SIZE = 32

# Replay memory
memory = deque(maxlen=MEMORY_SIZE)

# Crear red neuronal
state_size = env_cart.observation_space.shape[0]
action_size = env_cart.action_space.n

def build_model():
    # ✅ Forzamos conversión a int puro de Python
    s_size = int(state_size)
    a_size = int(action_size)

    model = Sequential([
        Input(shape=(s_size,)),
        Dense(24, activation='relu'),
        Dense(24, activation='relu'),
        Dense(a_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=LEARNING_RATE))
    return model

model = build_model()
target_model = build_model()
target_model.set_weights(model.get_weights())

print(f"\n🏗️  Arquitectura de la red neuronal:")
model.summary()

# Función para elegir acción
def choose_action_dqn(state, epsilon):
    if np.random.rand() <= epsilon:
        return env_cart.action_space.sample()
    q_values = model.predict(state.reshape(1, -1), verbose=0)
    return np.argmax(q_values[0])

# Entrenar DQN (optimizado)
print(f"\n🚀 Entrenando DQN por {EPISODES_DQN} episodios...")
rewards_dqn = []

for episode in range(EPISODES_DQN):
    state, _ = env_cart.reset()
    total_reward = 0
    done = False

    while not done:
        action = choose_action_dqn(state, EPSILON_DQN)
        next_state, reward, terminated, truncated, _ = env_cart.step(action)
        done = terminated or truncated

        # Guardar experiencia
        memory.append((state, action, reward, next_state, done))

        state = next_state
        total_reward += reward

        # Experience Replay (batch training - más eficiente)
        if len(memory) > BATCH_SIZE:
            batch = random.sample(memory, BATCH_SIZE)

            states = np.array([exp[0] for exp in batch])
            actions = np.array([exp[1] for exp in batch])
            rewards = np.array([exp[2] for exp in batch])
            next_states = np.array([exp[3] for exp in batch])
            dones = np.array([exp[4] for exp in batch])

            # Calcular targets
            next_q_values = target_model.predict(next_states, verbose=0)
            max_next_q = np.amax(next_q_values, axis=1)
            targets = rewards + (1 - dones) * GAMMA_DQN * max_next_q

            # Entrenar en batch
            current_q_values = model.predict(states, verbose=0)
            for i in range(BATCH_SIZE):
                current_q_values[i][actions[i]] = targets[i]

            model.fit(states, current_q_values, epochs=1, verbose=0)

    rewards_dqn.append(total_reward)

    # Actualizar target network cada 10 episodios
    if episode % 10 == 0:
        target_model.set_weights(model.get_weights())

    # Decaer epsilon
    if EPSILON_DQN > EPSILON_MIN_DQN:
        EPSILON_DQN *= EPSILON_DECAY_DQN

    # Progreso
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(rewards_dqn[-25:])
        print(f"   Episodio {episode+1}/{EPISODES_DQN} | Recompensa promedio: {avg_reward:.1f} | Epsilon: {EPSILON_DQN:.3f}")

print(f"\n✅ Entrenamiento DQN completado")

# ==========================================
# 10. VISUALIZAR APRENDIZAJE DQN
# ==========================================
print("\n" + "="*70)
print("📈 VISUALIZACIÓN DQN")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recompensa por episodio
axes[0].plot(rewards_dqn, alpha=0.3, color='blue', label='Recompensa por episodio')
if len(rewards_dqn) >= 25:
    axes[0].plot(np.convolve(rewards_dqn, np.ones(25)/25, mode='valid'),
                 color='red', linewidth=2, label='Promedio móvil (25)')
axes[0].axhline(y=200, color='green', linestyle='--', label='Meta: 200 pasos')
axes[0].set_title('🎢 CartPole: Recompensa por Episodio', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Episodio')
axes[0].set_ylabel('Recompensa (pasos)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribución de recompensas
axes[1].hist(rewards_dqn, bins=20, color='purple', edgecolor='black', alpha=0.7)
axes[1].axvline(x=np.mean(rewards_dqn), color='red', linestyle='--',
                label=f'Media: {np.mean(rewards_dqn):.1f}')
axes[1].set_title('📊 Distribución de Recompensas', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recompensa')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================
# 11. LECCIONES CLAVE
# ==========================================
print("\n" + "="*70)
print("💡 LECCIONES CLAVE DEL DÍA 26")
print("="*70)

print("""
1. Reinforcement Learning (RL):
   - El agente aprende por ensayo y error
   - No necesita datos etiquetados
   - Maximiza recompensa acumulada a largo plazo

2. Q-Learning (Tabular):
   - Usa una tabla Q para almacenar valores
   - Funciona bien con espacios de estado discretos y pequeños
   - Fórmula: Q(s,a) = Q(s,a) + α * [r + γ * max(Q(s',a')) - Q(s,a)]

3. DQN (Deep Q-Network):
   - Usa red neuronal para aproximar Q-values
   - Maneja espacios de estado continuos (infinitos)
   - Experience Replay: reutiliza experiencias pasadas
   - Target Network: estabiliza el entrenamiento

4. Exploración vs Explotación (ε-greedy):
   - ε alto: explorar acciones aleatorias (aprender)
   - ε bajo: explotar mejor acción conocida (ganar)
   - Decaer ε gradualmente para balancear ambos

5. Aplicaciones reales de RL:
   - Juegos (AlphaGo, AlphaStar)
   - Robots (caminar, agarrar objetos)
   - Autos autónomos
   - Trading algorítmico
   - Recomendación de contenido
  """)


print("\n✅ ¡DÍA 26 COMPLETADO! 🎮🧠")